## This is meant to be running directly on the RFSoC (No Labscript) for testing. Run the Kernel on the device via Jupyter Kernel = IP address

#### Ok now that that is out of the way on to the frequency sweep. 
The Goal is to do a low jitter function on the RFSoC which is a 6th term polynomial. However through looping as well as simplifying it to a bunch of differences we should be able to do a frequency sweep. Our goal is 109 MHz to 111 MHz in 100 us

In [ ]:
import os
os.environ.setdefault("BOARD", "ZCU216")
os.environ.setdefault("XILINX_XRT", "/usr")

import numpy as np
import matplotlib.pyplot as plt

from qick import QickSoc
from qick.asm_v2 import AveragerProgramV2, QickSweep1D

# tProc v2 firmware -- the only one available on this board (rest of the v2
# firmware/projects/.../out/ symlinks are broken build artifacts). Reserved
# for firmware testing/unstable in general, but fine for this notebook.
BITFILE = "/home/xilinx/jupyter_notebooks/amo_qick/tests/rf_board_firmware/d_1.bit"

# "Only the first channels work" on this bitstream -- confirmed via the
# printed board config: gen_ch 0 -> physical DAC port 0, ro_ch 10 -> physical
# ADC port 0. Wired DAC0<->ADC0 for loopback.
GEN_CH = 0
RO_CH = 10

soc = QickSoc(bitfile=BITFILE)
soccfg = soc
print(soc)


### Stage A — basic linear sweep validation (`QickSweep1D`)

Three-stage build-up (see the plan): plain `QickSweep1D` sweep first, then a chained sequence
of smaller sweeps, then (the real goal) a continuous chirp driven by manual per-iteration
register math. This is Stage A: validate the proven `AveragerProgramV2` + `QickSweep1D` +
`add_loop` pattern, unmodified in structure, on our actual channels/loopback wiring.

Three fixes were needed vs. the original snippet to make it actually run (none change the
core pattern):
- `declare_readout(ch=ro_ch, length=cfg['ro_len'])` — without it, `acquire_decimated()`
  returned an empty list.
- `add_readoutconfig(...)` + `send_readoutconfig(...)` — `ro_ch=10` is a dynamic readout
  (`axis_dyn_readout_v1`); without sending its demod config, `acquire_decimated()` hung
  indefinitely (confirmed: killed a process stuck at ~100% CPU for 5+ minutes).
- An explicit `self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])` in `_body` —
  `add_readoutconfig`/`send_readoutconfig` only sets the demod frequency, it doesn't arm the
  accumulator buffer. (`self.measure()` does not exist in this qick version's `asm_v2.py`,
  only in `asm_v1.py` — v1-only API, not usable here.)

`ro_freq=0` (no downconversion) so the decimated samples directly show the driving
oscillation itself, per your request, rather than a beat note against a nonzero demod
carrier.


In [ ]:
class SweepProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        frequency = QickSweep1D("loopedy_loop", cfg["f_start"], cfg["f_stop"])
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        self.declare_gen(ch=gen_ch, nqz=cfg['nqz'], mixer_freq=cfg['mixer_freq'], ro_ch=ro_ch)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch,
                       style="const",
                       freq=frequency,
                       length=cfg['flat_len'],
                       phase=cfg['phase'],
                       gain=cfg['gain'],
                      )
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['ro_freq'], gen_ch=gen_ch, phase=0)
        self.send_readoutconfig(ch=ro_ch, name="myro", t=0)
        self.add_loop("loopedy_loop", cfg['n_loop'], exec_before=None, exec_after=None)

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])


# n_loop (points across the sweep) and ro_len (samples/point -> resolution) are
# balanced against acquire_decimated's 4096-sample decimated-buffer cap:
# n_loop=8, ~492 samples/segment -> ~3936 total, resolution 307.2/492 ~= 0.62MHz
# (vs 9.9MHz with the original 31-sample/window snippet -- far too coarse to
# see a 2MHz-wide sweep at all).
DECIMATED_FS_MHZ = 307.2   # ro_ch=10 (dyn_readout) decimated sample rate
n_loop = 8
ro_len_us = 1.6            # -> round(1.6*307.2) = 492 samples/segment

config = {'gen_ch': GEN_CH,
          'ro_ch': RO_CH,
          'mixer_freq': 100,
          'f_start': 109,
          'f_stop': 111,
          'ro_freq': 0,   # no downconversion -- decimated samples directly show the
                          # driving oscillation (decimated rate 307.2MHz clears Nyquist
                          # for 111MHz), not a beat note
          'nqz': 1,
          'ro_len': ro_len_us,
          'flat_len': ro_len_us + 0.2,  # pulse must outlast the readout window
          'trig_time': 0.05,
          'phase': 0,
          'gain': 1.0,
          'n_loop': n_loop,
         }

prog = SweepProgram(soccfg, reps=1, final_delay=0.0, cfg=config)
print(prog)

iq_list = prog.acquire_decimated(soc, progress=True)
iq = np.asarray(iq_list[0])
print("acquire_decimated returned shape:", iq.shape)  # (n_loop, n_samples, 2)


In [ ]:
I_full = iq[..., 0].astype(float).ravel()
Q_full = iq[..., 1].astype(float).ravel()
mag_full = np.abs(I_full + 1j * Q_full)
t_axis = np.arange(len(I_full))
print(f"max|IQ|={mag_full.max():.1f}, mean|IQ|={mag_full.mean():.1f}, n_samples={len(I_full)}")

# --- Frequency analysis ---
# Ignore amplitude/envelope entirely -- only look at the driving frequency.
# ro_freq=0 means Q is trivially 0 too, so I is a real-valued capture -> use
# rfft (real FFT). Shape is (n_loop, n_samples, 2) -- one trigger/iteration.
I_trig0 = iq[:, :, 0].astype(float)   # (n_loop, n_samples)
n_samples = I_trig0.shape[1]
freqs_rfft = np.fft.rfftfreq(n_samples, d=1.0 / DECIMATED_FS_MHZ)

apparent_freqs = []
for row in I_trig0:
    spec = np.abs(np.fft.rfft(row - row.mean()))   # remove DC before peak search
    apparent_freqs.append(freqs_rfft[np.argmax(spec)])
apparent_freqs = np.array(apparent_freqs)
intended_freqs = np.linspace(config['f_start'], config['f_stop'], config['n_loop'])
print(f"samples/segment={n_samples}, resolution={DECIMATED_FS_MHZ/n_samples:.3f}MHz")
print("apparent (aliased) peak freqs:", np.round(apparent_freqs, 2))
print("intended freqs:               ", np.round(intended_freqs, 2))

fig, axes = plt.subplots(3, 1, figsize=(9, 10))
axes[0].plot(t_axis, I_full, label="I", lw=0.7)
axes[0].plot(t_axis, Q_full, label="Q", lw=0.7)
axes[0].legend(); axes[0].set_title("Loopback I/Q (acquire_decimated, raw)"); axes[0].set_xlabel("sample")

axes[1].plot(t_axis, mag_full, lw=0.7)
axes[1].set_title("Magnitude (amplitude/envelope -- ignore for freq analysis)"); axes[1].set_xlabel("sample")

axes[2].plot(range(config['n_loop']), apparent_freqs, "o-", label="apparent (aliased) peak freq")
ax2b = axes[2].twinx()
ax2b.plot(range(config['n_loop']), intended_freqs, "--", color="red", label="intended freq (MHz)")
axes[2].set_xlabel("loop iteration"); axes[2].set_ylabel("apparent aliased freq [MHz]")
ax2b.set_ylabel("intended freq [MHz]", color="red")
axes[2].set_title(f"Driving frequency vs. loop iteration (resolution ~{DECIMATED_FS_MHZ/n_samples:.2f}MHz)")
axes[2].legend(loc="upper left")
fig.tight_layout()


**Stage A: validated.** `max|IQ|≈3146`, real signal. Measured (aliased) peak frequencies per
loop iteration: `[109.27, 109.27, 109.27, 109.89, 109.89, 110.52, 110.52, 111.14]` MHz,
closely tracking the intended linear 109→111MHz trajectory (quantized to the ~0.62MHz FFT
resolution, but clearly monotonic and correct — not stuck, not an artifact).

Two earlier bugs, both traced to the same root cause and now fixed:
- **Resolution floor**: the original snippet's 31 samples/window gave ~9.9MHz-wide FFT bins,
  far coarser than the 2MHz sweep range. Fixed by balancing `n_loop` (points across the
  sweep) against `ro_len` (samples/point) against `acquire_decimated`'s 4096-sample buffer
  cap: `n_loop=8`, `ro_len=1.6us` (~492 samples/segment) → ~0.62MHz resolution.
- **Data dropout** (iterations 15-29 all-zero, in the earlier 30-iteration run): turned out to
  be a copy-paste bug — `_body` had been calling `self.trigger(...)` *twice* in the standalone
  debug script (not the notebook, which was always correct). Removing the duplicate fixed
  both the dropout and the earlier "2 triggers/iteration" oddity in one shot — same bug, not
  two separate issues.

A DDR4-buffer-based approach (for even more samples/segment) was tried and abandoned: it hung
indefinitely using the same low-level `config_all()`+`tproc.start()` bypass that works fine
for tProc v1 — v2's tProc apparently needs more startup sequencing than a single `start()`
register write provides, which the high-level `acquire_decimated()` already handles
internally. Since `acquire_decimated()` alone gets us a working, monotonic sweep measurement,
DDR4 wasn't pursued further.

**Next: Stage B** (chained sequence of smaller sweeps), then **Stage C** (continuous chirp via
manual per-iteration register math — the actual "low jitter" goal).


### Stage C — the low-jitter continuous chirp (the actual goal)

Stage B (a sweep chained from several smaller `QickSweep1D` segments) was skipped by request
once Stage A validated the channels/mechanism — Stage C's register-cascade approach subsumes
it anyway.

Stage A's `QickSweep1D` only supports a *constant* per-iteration step. Our real target
trajectory is a smoothstep `S(t)=3t²-2t³` (not a straight ramp), which needs a *varying*
per-step increment — so Stage C drives the frequency by hand instead, using v2's raw register
API: precompute the trajectory as DDS codes, take successive finite differences, then every
loop iteration cascade-update a chain of registers (`w_freq += d1; d1 += d2; d2 += d3`) in
ascending order so each step uses values not yet touched this iteration — the same Newton
forward-difference (DDA) technique, issued through v2's own instruction API.

Two mechanics had to be got right:
- `self.pulse()` replays the waveform from its stored *memory*, not the live `w0`-`w5`
  registers — so updates need `write_wmem` to persist, and `read_wmem` *first* to preserve
  the waveform's other fields (phase/gain/etc.) before overwriting just the frequency.
- `cascade_order=3` is not a truncation, it's *exact* for this trajectory: a cubic's true
  4th-and-higher finite differences are zero, so a 3-register cascade (with `d3` held
  constant) reconstructs it exactly. Going to order 4+ feeds `freq2reg`'s own quantization
  noise (the tiny nonzero `d4,d5,d6` that show up in practice) back into the cascade, which
  integrates into a spurious runaway high-degree polynomial rather than adding fidelity —
  full derivation and the bisection that found this in `FreqSweep_Analysis.ipynb`.

Also includes the zero-gain `dummypulse` flush (a documented QICK pattern — a 3-fabric-cycle,
gain=0 pulse) right after each segment's real pulse: drives the generator cleanly to zero
before the next segment's frequency change, removing a ~0.4us transient glitch that was
otherwise present right at each segment boundary (diagnosed and confirmed fixed via
`boundary_capture.py`, see `FreqSweep_Analysis.ipynb`).


In [ ]:
from qick.asm_v2 import QickParam


def smoothstep_trajectory(soc, gen_ch, f0, f1, n):
    """Cubic smoothstep S(t)=3t^2-2t^3 (S(0)=0,S(1)=1,S'(0)=S'(1)=0)."""
    t = np.linspace(0, 1, n)
    S = 3 * t**2 - 2 * t**3
    f_mhz = f0 + (f1 - f0) * S
    codes = np.array([soc.freq2reg(f, gen_ch=gen_ch) for f in f_mhz], dtype=np.int64)
    diffs = [codes]
    for _ in range(3):
        diffs.append(np.diff(diffs[-1]))
    deltas0 = [int(d[0]) for d in diffs]  # [f0_code, d1[0], d2[0], d3[0]]
    print(f"[traj] f_mhz[0]={f_mhz[0]:.6f} f_mhz[-1]={f_mhz[-1]:.6f} deltas0={deltas0}")
    return f_mhz, deltas0


class ChirpProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        gen_ch = cfg['gen_ch']
        ro_ch = cfg['ro_ch']
        deltas0 = cfg['deltas0']

        self.declare_gen(ch=gen_ch, nqz=1, mixer_freq=cfg['mixer_freq'], ro_ch=ro_ch)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch, style="const",
                       freq=cfg['f0_mhz'], length=cfg['flat_len'], phase=0, gain=cfg['gain'])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['ro_freq'], gen_ch=gen_ch, phase=0)
        self.send_readoutconfig(ch=ro_ch, name="myro", t=0)

        for k in range(1, 4):
            self.add_reg(name=f'd{k}', init=QickParam(deltas0[k]))

        self.add_loop("chirp_loop", cfg['n_loop'], exec_before=None, exec_after=None)

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])
        # zero-gain flush -- removes the ~0.4us boundary transient (see markdown above)
        self.pulse(ch=cfg['gen_ch'], name="dummypulse", t="auto")
        # cascading Newton forward-difference update, ascending order (each step
        # reads a register not yet touched this iteration, so old values are
        # used throughout -- matches the correct recurrence). cascade_order=3
        # is exact for this cubic trajectory (see markdown above).
        self.read_wmem("mypulse_w0")
        self.inc_reg('w_freq', 'd1')
        self.inc_reg('d1', 'd2')
        self.inc_reg('d2', 'd3')
        self.write_wmem("mypulse_w0")


F0_MHZ, F1_MHZ = 109.0, 111.0
N_LOOP = 100            # segments -> 100 * 1.0us = 100us total
SEG_LEN_US = 1.0
RO_LEN_US = 0.13        # -> ~40 samples/segment
TRIG_TIME_US = 0.65     # same pipeline-transient fix as Stage A

f_mhz_traj, deltas0 = smoothstep_trajectory(soc, GEN_CH, F0_MHZ, F1_MHZ, N_LOOP)

config = {
    'gen_ch': GEN_CH, 'ro_ch': RO_CH, 'mixer_freq': 100,
    'f0_mhz': F0_MHZ, 'deltas0': deltas0,
    'ro_freq': 0, 'ro_len': RO_LEN_US, 'flat_len': SEG_LEN_US,
    'trig_time': TRIG_TIME_US, 'gain': 1.0, 'n_loop': N_LOOP,
}

chirp_prog = ChirpProgram(soccfg, reps=1, final_delay=0.0, cfg=config)
print(chirp_prog)

chirp_iq_list = chirp_prog.acquire_decimated(soc, progress=True)
chirp_iq = np.asarray(chirp_iq_list[0])
print("acquire_decimated returned shape:", chirp_iq.shape)   # (n_loop, n_samples, 2)


In [ ]:
def parabolic_interp(spec, k):
    """Quadratic interpolation around FFT bin k for sub-bin peak location."""
    if k <= 0 or k >= len(spec) - 1:
        return float(k)
    alpha, beta, gamma = spec[k - 1], spec[k], spec[k + 1]
    denom = alpha - 2 * beta + gamma
    if denom == 0:
        return float(k)
    return k + 0.5 * (alpha - gamma) / denom


def windowed_interp_freq(row, fs):
    """Hann-windowed rfft + parabolic sub-bin interpolation -> frequency (MHz).
    Needed here (vs. Stage A's plain argmax) because each chirp segment only has
    ~40 samples -> a bare FFT bin is ~7.7MHz wide, far coarser than the whole
    2MHz sweep range."""
    n = len(row)
    hann = np.hanning(n)
    spec = np.abs(np.fft.rfft((row - row.mean()) * hann))
    k = int(np.argmax(spec))
    return parabolic_interp(spec, k) * (fs / n)


chirp_I_seg = chirp_iq[:, :, 0].astype(float)
chirp_Q_seg = chirp_iq[:, :, 1].astype(float)
chirp_mag_full = np.abs(chirp_I_seg.ravel() + 1j * chirp_Q_seg.ravel())
chirp_t_us = np.arange(chirp_I_seg.size) / DECIMATED_FS_MHZ
print(f"max|IQ|={chirp_mag_full.max():.1f}, mean|IQ|={chirp_mag_full.mean():.1f}, shape={chirp_I_seg.shape}")

chirp_apparent_freqs = np.array([windowed_interp_freq(row, DECIMATED_FS_MHZ) for row in chirp_I_seg])
chirp_idx = np.arange(N_LOOP)
chirp_seg_center_us = (chirp_idx + 0.5) * SEG_LEN_US

chirp_slope, chirp_intercept = np.polyfit(chirp_idx, chirp_apparent_freqs, 1)
chirp_fit = chirp_slope * chirp_idx + chirp_intercept
chirp_resid = chirp_apparent_freqs - chirp_fit
chirp_r2 = 1 - np.sum(chirp_resid**2) / np.sum((chirp_apparent_freqs - chirp_apparent_freqs.mean())**2)
print(f"slope={chirp_slope:.4f}MHz/iter, R^2={chirp_r2:.5f}, residual std={chirp_resid.std():.4f}MHz")

fig, axes = plt.subplots(2, 1, figsize=(9, 8))
axes[0].plot(chirp_t_us, chirp_mag_full, lw=0.5)
axes[0].set_title("Magnitude"); axes[0].set_xlabel("time [us]"); axes[0].set_ylabel("magnitude [a.u.]")

axes[1].plot(chirp_seg_center_us, chirp_apparent_freqs, "o", ms=3, label="measured")
axes[1].plot(chirp_seg_center_us, f_mhz_traj, "--", color="red", lw=1.5, label="intended (smoothstep)")
axes[1].set_xlabel("time [us]"); axes[1].set_ylabel("Frequency [MHz]")
axes[1].set_title("Low-jitter chirp: measured vs. intended"); axes[1].legend()
fig.tight_layout()


**Stage C: validated.** `cascade_order=3`, 109→111MHz smoothstep over 100 segments × 1.0us:
**R²=0.98493**, residual std **0.093MHz**, a clean monotonic S-curve tracking the intended
smoothstep shape (small offset from the nominal endpoints is the same systematic
frequency-register-quantization effect seen in Stage A's constant-frequency check, not
noise). This is the final answer to the original goal: a continuous, FPGA-timed, low-jitter
frequency chirp produced entirely by manual tProc v2 register math — no `QickSweep1D`
involved, cascading exactly as many finite-difference registers (3) as this trajectory's
polynomial order actually requires.

Full derivation of why order 3 (not the originally-planned 6), the `read_wmem`/`write_wmem`
mechanics, decoupled hardware/analysis workflow, and further diagnostics (pulse-to-pulse
boundary captures, zoomed transition-slope) live in `FreqSweep_Analysis.ipynb`, which runs
entirely off-board against the raw I/Q this notebook (and its acquire-only script
counterparts in `python/`) save to `json/`/`csv/`.


### Stage F — UNTESTED: finer-resolution 8-tone spectrum via a PFB readout channel

**⚠️ NOT YET RUN ON HARDWARE.** Written while the board was disconnected, to be ready for
next time — do not trust anything below until it's actually been run and checked.

Stage D's 8-tone comb (see `FreqSweep_Analysis.ipynb`) could only be resolved to ~12MHz-wide
FFT bins per tone (25 samples/tone at `RO_CH=10`'s fixed 307.2MHz decimated rate) — far too
coarse to separate the 1MHz-spaced tones as distinct peaks. FFT resolution is `Δf=fs/N` (total
*real time* spanned by one continuous capture, not sample count alone) — `RO_CH=10`'s
307.2MHz decimated rate looks like a fixed hardware property (not software-adjustable), but
the board's printed config also lists 8 `axis_pfb_readout_v4` channels (indices 2-9) with a
natively lower decimated rate: **38.4MHz, 8x lower** — so the same sample budget spans 8x more
real time, giving 8x finer resolution (enough to cleanly resolve the comb, with margin).

**Two things this has NOT verified — check both before trusting anything else:**
1. **Physical wiring**: the PFB channels are wired to ADC port 5 (per the printed config),
   not ADC port 0 where the DAC0↔ADC0 loopback has lived for every prior stage. The loopback
   cable needs to move to ADC port 5 first.
2. **Software config path**: `RO_CH=10` is "configured by tProc" per-program via the standard
   `declare_readout`/`add_readoutconfig`/`send_readoutconfig` calls used throughout. The
   printed config listed the PFB channels as "configured by PYNQ" instead — possibly meaning
   they need `qick.py`'s lower-level `config_mux_readout(pfbpath, cfgs)` call instead. This
   notebook guesses the standard API works transparently for any readout type — unconfirmed.
   If the probe cell below hangs, that's the likely cause (same class of issue as Stage A's
   original multi-minute hang from a missing readout config) — don't wait more than ~30s
   before interrupting the kernel if it does.

Also unverified: the PFB readout's own internal buffer limit vs. the confirmed 4096-sample
`acquire_decimated` cap seen on `RO_CH=10` — the printed config showed different PFB channels
with different buffer depths (2-5: 1024 samples; 6-9: 4096 samples). `RO_CH=6` below is a
first guess (one of the 4096-depth ones) — re-check against a fresh `print(soc)` first, since
firmware may have changed.

**Run the probe cell first, alone, and compare its output against the assumptions above
before running the acquisition cell.**


In [ ]:
# UNTESTED -- run this cell ALONE first. Compare printed config against the
# assumptions in the markdown above (RO_CH, decimated rate, buffer depth,
# ADC port) before running anything else in this section.
RO_CH_PFB_GUESS = 6   # one of the two candidate channels with the larger
                       # (4096-sample) decimated buffer per the printed config
                       # (channels 6,7,8,9 all looked equivalent; 2,3,4,5 only
                       # have a 1024-sample buffer)

print(f"=== soc['readouts'][{RO_CH_PFB_GUESS}] ===")
print(soc['readouts'][RO_CH_PFB_GUESS])


In [ ]:
# UNTESTED -- only run after the probe cell above looks right AND the
# loopback cable has been moved to ADC port 5. See python/stageF_finer_readout_probe.py
# for the standalone hardware-script version of this same code (identical
# logic, meant for the decoupled acquire-then-analyze workflow).

RO_CH_F = RO_CH_PFB_GUESS
DECIMATED_FS_MHZ_F = 38.4     # UNVERIFIED -- confirm against the probe cell's
                                # printed output, not just this comment
TONE_STARTS_MHZ_F = [106.0, 107.0, 108.0, 109.0, 110.0, 111.0, 112.0, 113.0]
N_TONES_F = len(TONE_STARTS_MHZ_F)
SEG_LEN_US_F = 15.0     # UNVERIFIED -- generous guess, revisit once the real
                         # buffer cap for this readout is confirmed
RO_LEN_US_F = 13.0       # -> ~500 samples/tone if DECIMATED_FS_MHZ_F is right
                         # (500*8=4000 total -- UNCONFIRMED cap for this channel)
TRIG_TIME_US_F = 1.5    # UNVERIFIED -- the ~0.4-0.65us transient was
                         # characterized on RO_CH=10 only; this channel's own
                         # pipeline latency has never been measured
DUMMY_LEN_US_F = 3 / 599.04  # dummypulse fixed 3-fabric-cycle length (gen0 fabric clock)


class FinerReadoutProgram(AveragerProgramV2):
    """Static (non-swept) 8-tone capture -- same per-tone pulse+trigger+flush
    structure as Stage D, minus the sweep/register-cascade (goal here is just
    a resolvable spectrum). If this works, Stage D's cascade could be grafted
    back on the same way it was added to Stage C's mechanism."""
    def _initialize(self, cfg):
        gen_ch = cfg['gen_ch']; ro_ch = cfg['ro_ch']
        self.declare_gen(ch=gen_ch, nqz=1, mixer_freq=cfg['mixer_freq'], ro_ch=ro_ch)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        for k, f0 in enumerate(cfg['tone_freqs']):
            self.add_pulse(ch=gen_ch, name=f"tone{k}", ro_ch=ro_ch, style="const",
                           freq=f0, length=cfg['flat_len'], phase=0, gain=cfg['gain'])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['ro_freq'], gen_ch=gen_ch, phase=0)
        self.send_readoutconfig(ch=ro_ch, name="myro", t=0)

    def _body(self, cfg):
        t = 0.0
        for k in range(cfg['n_tones']):
            self.pulse(ch=cfg['gen_ch'], name=f"tone{k}", t=t)
            self.trigger(ros=[cfg['ro_ch']], pins=[0], t=t + cfg['trig_time'])
            self.pulse(ch=cfg['gen_ch'], name="dummypulse", t=t + cfg['flat_len'])
            t += cfg['flat_len'] + DUMMY_LEN_US_F


config_f = {
    'gen_ch': GEN_CH, 'ro_ch': RO_CH_F, 'mixer_freq': 100,
    'tone_freqs': TONE_STARTS_MHZ_F, 'n_tones': N_TONES_F,
    'ro_freq': 0, 'ro_len': RO_LEN_US_F, 'flat_len': SEG_LEN_US_F,
    'trig_time': TRIG_TIME_US_F, 'gain': 1.0,
}

finerro_prog = FinerReadoutProgram(soccfg, reps=1, final_delay=0.0, cfg=config_f)
print(finerro_prog)   # inspect the ASM before trusting acquire_decimated --
                       # same habit that caught real bugs in Stage D before it
                       # ever ran on hardware

finerro_iq_list = finerro_prog.acquire_decimated(soc, progress=True)
finerro_iq = np.asarray(finerro_iq_list[0])
print("acquire_decimated returned shape:", finerro_iq.shape)
# Expect (n_tones, n_samples, 2) if the shape convention matches Stage D's
# (acquire_decimated groups by triggers-per-rep) -- verify this explicitly,
# don't assume, exactly the kind of thing that was wrong in the Stage D draft
# before it was tested.


In [ ]:
# UNTESTED -- adjust indexing here if the actual returned shape from the
# previous cell doesn't match (n_tones, n_samples, 2); don't assume.
finerro_I = finerro_iq[:, :, 0].astype(float)   # (n_tones, n_samples)
finerro_Q = finerro_iq[:, :, 1].astype(float)
n_samples_f = finerro_I.shape[1]
print(f"n_samples/tone={n_samples_f}, resolution={DECIMATED_FS_MHZ_F/n_samples_f:.4f}MHz "
      f"(vs Stage D's ~12.3MHz -- this should be the whole point of Stage F)")

fig, ax = plt.subplots(figsize=(12, 5))
colors_f = plt.cm.viridis(np.linspace(0, 1, N_TONES_F))
for k in range(N_TONES_F):
    row = finerro_I[k]
    hann = np.hanning(len(row))
    spec = np.abs(np.fft.rfft((row - row.mean()) * hann))
    freqs = np.fft.rfftfreq(len(row), d=1 / DECIMATED_FS_MHZ_F)
    ax.plot(freqs, spec, color=colors_f[k], lw=1.2, label=f"tone{k} ({TONE_STARTS_MHZ_F[k]:.0f}MHz)")
ax.set_xlim(100, 120)
ax.set_xlabel("freq [MHz]")
ax.set_ylabel("|FFT|")
ax.legend(fontsize=8, ncol=2)
ax.set_title("Stage F (UNTESTED): 8-tone spectrum via PFB readout -- "
              "check whether peaks are now actually separated")
fig.tight_layout()


### Stage G — UNTESTED: engineered AM-sideband comb, genuinely simultaneous 8 tones

**⚠️ NOT YET RUN ON HARDWARE.** Written while the board was disconnected, to be ready for
next time — do not trust anything below until it's actually been run and checked.

Every multitone approach tried so far (Stage D's time-multiplexing, Stage E's TDM-comb
hypothesis) produces the 8 tones *sequentially*, with switching artifacts as an unavoidable
side effect. This is a different idea: keep the DDS carrier **fixed** at the center
(109.5MHz) and multiply it by a custom **gain envelope** built from exactly 4 cosines (0.5,
1.5, 2.5, 3.5MHz). Real amplitude modulation of a real carrier produces both upper *and*
lower sidebands for each modulation component, so `109.5 ± {0.5,1.5,2.5,3.5} =
106,107,108,109,110,111,112,113` — all 8 target tones, genuinely **simultaneous**, by direct
Fourier construction rather than switching debris. Sweeping the whole comb is then just
sweeping the *carrier* register with the exact same cascade mechanism as Stage C.

**Math validated by pure simulation before writing any of this** — see
`FreqSweep_Analysis.ipynb`'s "Stage G design" section: ~78dB suppression of everything except
the 8 target tones, dramatically cleaner than Stage D or Stage E. The hardware side is new,
though — none of it has touched a real generator.

**What this has NOT verified — check all of these before trusting it:**
1. `add_envelope()`'s exact signature/units — assumed here to be
   `prog.add_envelope(ch=gen_ch, name=..., idata=<int16 array>, qdata=<int16 array>)`, scaled
   to gen0's `MAXV=32766`, with length a multiple of `samps_per_clk=16`. Standard QICK v2 API
   from general knowledge, not re-confirmed against this board's `qick_asm.py` source this
   session — read its `add_envelope` docstring first if this errors.
2. Whether the `read_wmem`/`inc_reg`/`write_wmem` register-cascade sweep (proven for
   `const`-style pulses in Stage C) behaves identically for `arb`-style pulses. The Waveform
   structure looked style-agnostic in every ASM dump seen this session, but genuinely
   unverified — inspect `print(prog)` before trusting `acquire_decimated`.
3. `trig_time`/pipeline-transient behavior for this pulse type — never characterized
   (`boundary_capture.py`'s measurement was `const`-style only). Guessed at `0.7us` below.
4. Whether `qdata=0` (real-only envelope) is the right I/Q convention for gen0's
   `complex_env=True` architecture — the simulation validates the *math*, not this specific
   hardware mapping.

See `python/stageG_am_comb_sweep.py` for the standalone hardware-script version of this same
code (identical logic, decoupled acquire-then-analyze workflow).


In [ ]:
# UNTESTED -- see markdown above for what to verify before trusting this.

DAC_FS_MHZ_G = 9584.64      # gen0's real DAC sample rate -- envelope table rate
SAMPS_PER_CLK_G = 16         # gen0's envelope-length granularity (UNVERIFIED)
F_CENTER_MHZ_G = 109.5       # carrier center -- sweeps to 110.5 (comb: 107-114)
MOD_OFFSETS_MHZ_G = [0.5, 1.5, 2.5, 3.5]   # -> 8 sidebands at center +/- these
SPAN_MHZ_G = 1.0
N_STEPS_G = 50                # -> 50 * ~2us = 100us total
TRIG_TIME_US_G = 0.7          # UNVERIFIED GUESS -- see markdown point 3


def build_envelope_g(dac_fs_mhz, mod_offsets_mhz, samps_per_clk, maxv):
    """Exactly the 4 needed cosines, nothing else -- validated by simulation
    in FreqSweep_Analysis.ipynb. Rounds envelope length to a multiple of
    samps_per_clk, then re-derives the modulation frequencies as exact
    harmonics of the resulting discretized period so the table is perfectly
    periodic (no wraparound discontinuity)."""
    t_mod_us_nominal = 1.0 / mod_offsets_mhz[0]
    n_env = int(round(t_mod_us_nominal * dac_fs_mhz / samps_per_clk) * samps_per_clk)
    fundamental_mhz = dac_fs_mhz / n_env
    harmonics = [fundamental_mhz * (2 * k + 1) for k in range(len(mod_offsets_mhz))]
    t_env = np.arange(n_env) / dac_fs_mhz
    m = sum(np.cos(2 * np.pi * f * t_env) for f in harmonics)
    m = m / np.max(np.abs(m))
    idata = np.round(m * maxv).astype(np.int16)
    qdata = np.zeros(n_env, dtype=np.int16)
    t_mod_us_actual = n_env / dac_fs_mhz
    print(f"[env] n_env={n_env}, t_mod_us={t_mod_us_actual:.6f} (nominal {t_mod_us_nominal}), "
          f"harmonics={harmonics}")
    return idata, qdata, t_mod_us_actual


class AmCombProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        gen_ch = cfg['gen_ch']
        ro_ch = cfg['ro_ch']
        deltas0 = cfg['deltas0']

        self.declare_gen(ch=gen_ch, nqz=1, mixer_freq=cfg['mixer_freq'], ro_ch=ro_ch)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        # UNVERIFIED API -- see markdown point 1
        self.add_envelope(ch=gen_ch, name="combenv", idata=cfg['idata'], qdata=cfg['qdata'])
        self.add_pulse(ch=gen_ch, name="combpulse", ro_ch=ro_ch, style="arb",
                       envelope="combenv", freq=cfg['f0_mhz'], length=cfg['t_mod_us'],
                       phase=0, gain=cfg['gain'])

        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['ro_freq'], gen_ch=gen_ch, phase=0)
        self.send_readoutconfig(ch=ro_ch, name="myro", t=0)

        for k in range(1, 4):
            self.add_reg(name=f'd{k}', init=QickParam(deltas0[k]))

        self.add_loop("comb_loop", cfg['n_steps'], exec_before=None, exec_after=None)

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="combpulse", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])
        self.pulse(ch=cfg['gen_ch'], name="dummypulse", t="auto")
        # same cascade recurrence as Stage C, applied to the CENTER frequency
        # only -- the envelope (hence relative tone spacing) never changes
        self.read_wmem("combpulse_w0")
        self.inc_reg('w_freq', 'd1')
        self.inc_reg('d1', 'd2')
        self.inc_reg('d2', 'd3')
        self.write_wmem("combpulse_w0")


maxv_g = 32766   # gen0's MAXV, confirmed from earlier ASM dumps this session
idata_g, qdata_g, t_mod_us_g = build_envelope_g(DAC_FS_MHZ_G, MOD_OFFSETS_MHZ_G, SAMPS_PER_CLK_G, maxv_g)
f_mhz_traj_g, deltas0_g = smoothstep_trajectory(soc, GEN_CH, F_CENTER_MHZ_G, F_CENTER_MHZ_G + SPAN_MHZ_G, N_STEPS_G)

config_g = {
    'gen_ch': GEN_CH, 'ro_ch': RO_CH, 'mixer_freq': 100,
    'f0_mhz': F_CENTER_MHZ_G, 'deltas0': deltas0_g,
    'idata': idata_g, 'qdata': qdata_g, 't_mod_us': t_mod_us_g,
    'ro_freq': 0, 'ro_len': min(t_mod_us_g - 0.1, 1.9), 'trig_time': TRIG_TIME_US_G,
    'gain': 1.0, 'n_steps': N_STEPS_G,
}

amcomb_prog = AmCombProgram(soccfg, reps=1, final_delay=0.0, cfg=config_g)
print(amcomb_prog)   # inspect ASM before trusting acquire_decimated -- confirm the
                      # w_freq cascade actually targets the arb pulse's freq register
                      # the same way it did for Stage C's const pulse

amcomb_iq_list = amcomb_prog.acquire_decimated(soc, progress=True)
amcomb_iq = np.asarray(amcomb_iq_list[0])
print("acquire_decimated returned shape:", amcomb_iq.shape)   # expect (n_steps, n_samples, 2)


In [ ]:
# UNTESTED -- adjust indexing here if the actual returned shape from the
# previous cell doesn't match (n_steps, n_samples, 2); don't assume.
amcomb_I = amcomb_iq[:, :, 0].astype(float)
amcomb_Q = amcomb_iq[:, :, 1].astype(float)
n_samples_g = amcomb_I.shape[1]
print(f"n_samples/step={n_samples_g}, resolution={DECIMATED_FS_MHZ/n_samples_g:.4f}MHz")

# first step's spectrum -- if this actually works, expect 8 distinct sharp
# peaks near 106-113 (not a merged hump like every switching-based attempt)
row0 = amcomb_I[0]
hann = np.hanning(len(row0))
spec0 = np.abs(np.fft.rfft((row0 - row0.mean()) * hann))
freqs0 = np.fft.rfftfreq(len(row0), d=1 / DECIMATED_FS_MHZ)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freqs0, spec0, lw=1.2, color="steelblue")
targets_g = sorted([F_CENTER_MHZ_G + o for o in MOD_OFFSETS_MHZ_G] + [F_CENTER_MHZ_G - o for o in MOD_OFFSETS_MHZ_G])
for f0 in targets_g:
    ax.axvline(f0, color="red", ls="--", lw=0.8)
    ax.annotate(f"{f0:.0f}", xy=(f0, spec0.max() * 0.95), ha="center", fontsize=8, color="red")
ax.set_xlim(95, 125)
ax.set_xlabel("freq [MHz]")
ax.set_ylabel("|FFT|")
ax.set_title("Stage G (UNTESTED), step 0: does this actually give 8 distinct simultaneous peaks?")
fig.tight_layout()
